# 全量 718 组 BCSC 高置信度信道实验

本 Notebook 是正式实验的可复现入口与验收界面。流程严格为：先在临时目录做单组、低 Bootstrap 次数的最小规模验证；验证通过后，只执行一次 718 组正式主实验；随后执行固定分层 100 组参数敏感性实验；最后只读聚合报告。

当前正式主实验已完成 718/718，入模 click 为 6,545，敏感性实验已完成 100/100。为避免误触第二次全量拟合，本 Notebook 默认锁定计算阶段，只允许状态检查与只读汇总。原有 50 组实验未修改。

In [ ]:
from pathlib import Path
import csv
import json
import subprocess

PROJECT_ROOT = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
NOTEBOOK_ROOT = PROJECT_ROOT / "20"
OUTPUT_ROOT = NOTEBOOK_ROOT / "full_dataset_experiment"
EXPERIMENT_SCRIPT = NOTEBOOK_ROOT / "full_dataset_bcsc_experiment.py"
FINALIZER = NOTEBOOK_ROOT / "finalize_full_dataset_experiment.py"
PYTHON = Path(r"D:\Python3_13\python.exe")
for path in (OUTPUT_ROOT, EXPERIMENT_SCRIPT, FINALIZER, PYTHON):
    assert path.exists(), path
print(f"结果目录：{OUTPUT_ROOT}")

## 最小规模验证记录

正式运行前已完成两级验证：

1. 单个 PulseTrain、单/联合方法各 1 次 Bootstrap：验证 718 组发现逻辑、6,545 click 计数、单位 L2 范数、带宽约束、路径表和检查点 JSON。
2. 单个 PulseTrain、全部 12 个敏感性配置各 1 次 Bootstrap：验证 Jaccard、时延/幅值变化、Wilson 区间、投影触发率和参数结论聚合。

两项验证均写入临时目录或内存，没有占用正式检查点，也没有形成第二次 718 组实验。下列单元复核正式结果中的同类不变量。

In [ ]:
validation = json.loads((OUTPUT_ROOT / "acceptance_validation.json").read_text(encoding="utf-8"))
assert validation["accepted"], validation
assert validation["checks"]["formal_full_fit_executed_once"]
assert validation["checks"]["completed_718_pulsetrains"]
assert validation["checks"]["modeled_clicks_equal_6545"]
assert validation["checks"]["sensitivity_completed_100"]
print(json.dumps(validation, ensure_ascii=False, indent=2))

## 正式计算锁

正式 718 组实验已经完成一次。以下单元只显示状态，不会调用求解器。`full_dataset_bcsc_experiment.py` 也会在检测到已通过验收时拒绝再次运行主实验或敏感性实验。

In [ ]:
main_checkpoints = len(list((OUTPUT_ROOT / "checkpoints").glob("*.json")))
sensitivity_checkpoints = len(list((OUTPUT_ROOT / "sensitivity_checkpoints").glob("*.json")))
print(f"正式主实验检查点：{main_checkpoints}/718")
print(f"敏感性检查点：{sensitivity_checkpoints}/100")
assert main_checkpoints == 718 and sensitivity_checkpoints == 100
print("全量拟合已锁定；不会再次执行。")

## 只读重新聚合（可选）

如果只需要从既有检查点重新生成 CSV、NPZ、718 组图、HTML 报告和验收文件，可运行下一单元。它不会调用 BCSC。

In [ ]:
REBUILD_REPORT = False
if REBUILD_REPORT:
    subprocess.run([str(PYTHON), str(FINALIZER)], cwd=NOTEBOOK_ROOT, check=True)
else:
    print("现有报告已通过验收；未重复聚合。将 REBUILD_REPORT 改为 True 可只读重建报告。")

In [ ]:
with (OUTPUT_ROOT / "pulsetrain_summary.csv").open(encoding="utf-8-sig") as handle:
    pulse_trains = list(csv.DictReader(handle))
with (OUTPUT_ROOT / "high_confidence_channels.csv").open(encoding="utf-8-sig") as handle:
    high_channels = list(csv.DictReader(handle))
print(f"PulseTrain={len(pulse_trains)}, 入模click={sum(int(row['used_clicks']) for row in pulse_trains)}")
print(f"高置信度非直达径={len(high_channels)}")
print(f"报告：{OUTPUT_ROOT / 'report.html'}")